# Bước 1: Cài đặt thư viện
Chạy ô này để cài đặt thư viện giao tiếp với Azure.

In [ ]:
!pip install azure-storage-blob

# Bước 2: Khai báo cấu hình
Điền thông tin kết nối Azure của bạn vào đây.

In [ ]:
AZURE_CONNECTION_STRING = "DefaultEndpointsProtocol=https;AccountName=aicvideos;AccountKey=ZLdH5hcKb213WiCzOLCmfLrPRWR3rVkDdlfwV5iXr2snbY9lAMAJc9ZlHm9salxNTAvhcaNVJh0N+ASt+nNulQ==;EndpointSuffix=core.windows.net"
CONTAINER_NAME = "video"

# Đường dẫn tĩnh Batch 1 mà bạn đang dùng
TARGET_DIR = "/kaggle/input/datasets/aresusayhi/ai-challenge-2025/Videos/Videos"

# Nơi lưu file tạm trước khi up lên Azure (sẽ bị xóa ngay sau khi up để trống ổ)
TEMP_DIR = "/kaggle/working/proxy_videos"

# Bước 3: Hàm xử lý FFmpeg (Fast Start)
Hàm này copy giữ nguyên 100% chất lượng gốc, dung lượng gốc. Chỉ chèn cờ Fast Start.

In [ ]:
import os
import subprocess
from pathlib import Path

Path(TEMP_DIR).mkdir(parents=True, exist_ok=True)

def process_video_ffmpeg(input_path, output_path):
    """
    Giữ nguyên 100% chất lượng gốc (không giảm dung lượng, không giảm độ phân giải).
    Chỉ chèn thêm cờ faststart để sửa lỗi tua video bị đơ.
    """
    command = [
        "ffmpeg", "-y",             # -y: Ghi đè file nếu có
        "-i", str(input_path),
        "-c", "copy",               # Copy y nguyên luồng hình ảnh/âm thanh gốc, KHÔNG nén lại
        "-movflags", "faststart",   # TỐI QUAN TRỌNG: Dời mục lục lên đầu để tua nhanh
        str(output_path)
    ]
    
    try:
        subprocess.run(command, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
        return True
    except subprocess.CalledProcessError as e:
        print(f"Lỗi FFmpeg: {e.stderr.decode('utf-8', errors='ignore')}")
        return False

# Bước 4: Khởi chạy Upload hàng loạt (Hỗ trợ cả .mov)
Truy tìm thư mục con, ghim Fast Start và đẩy thẳng tắp lên Azure.

In [ ]:
from azure.storage.blob import BlobServiceClient
import time
import glob

# Khởi tạo kết nối Azure
blob_service_client = BlobServiceClient.from_connection_string(AZURE_CONNECTION_STRING)
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

print("1. Đang truy tìm ổ nhóm MP4/MOV...")
vid_dirs = set()
# Quét siêu tốc tìm chính xác cái thư mục cuối cùng chứa file video
for root, dirs, files in os.walk(TARGET_DIR):
    for f in files:
        if f.lower().endswith(('.mp4', '.mov')):
            vid_dirs.add(root)
            break

print(f"👉 Khóa mục tiêu thành công: {len(vid_dirs)} thư mục con. BẮT ĐẦU XỬ LÝ & BƠM LÊN AZURE...\n")
print("==========================================================================")

for i, directory in enumerate(vid_dirs, 1):
    print(f"\nĐang xử lý thư mục {i}/{len(vid_dirs)}: {directory}")
    
    video_files = []
    for ext in ('*.mp4', '*.mov', '*.MP4', '*.MOV'):
        video_files.extend(glob.glob(os.path.join(directory, ext)))
    video_files.sort()
    
    for v_idx, input_file in enumerate(video_files, 1):
        video_name = os.path.basename(input_file)
        
        # Đảm bảo file up lên luôn có đuôi .mp4 để web dễ đọc
        out_name = os.path.splitext(video_name)[0] + ".mp4"
        temp_file = os.path.join(TEMP_DIR, out_name)
        
        print(f"  [{v_idx}/{len(video_files)}] Đính Fast Start: {video_name} ... ", end="", flush=True)
        start_time = time.time()
        
        # 1. Đính Fast Start
        success = process_video_ffmpeg(input_file, temp_file)
        if not success:
            print("LỖI")
            continue
            
        # 2. Upload lên Azure
        blob_client = container_client.get_blob_client(out_name)
        with open(temp_file, "rb") as data:
            blob_client.upload_blob(data, overwrite=True)
            
        # 3. Xóa rác, giải phóng ổ cứng Kaggle
        os.remove(temp_file)
        
        elapsed = time.time() - start_time
        print(f"XONG ({elapsed:.1f}s)")

print("\n🎉 HOÀN TẤT TOÀN BỘ! BẠN QUA AZURE F5 LÀ THẤY FILE ĐÃ CHUẨN XỊN RỒI ĐÓ!")